## Predicting Revenue from Online Shopper Session Features

### Load dataset
#### Source - [online_shoppers_purchase_intention_dataset](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset)

# Milestone 1: Data Pipeline

In [ ]:
# Load the Online Shoppers dataset
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

file_path = Path(r"/content/online_shoppers_intention.csv")
df = pd.read_csv(file_path)

# Plotting configuration
sns.set_style("whitegrid")

# Show the dataset
print("Dataset shape:", df.shape)      # expected: (12330, 18)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 10 rows:")
display(df.head(10))


### Correlation

In [ ]:
# Convert boolean columns to 0/1
df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)

# Use only numeric columns for correlation
numeric_df = df.select_dtypes(include=["int64", "float64"])

# Correlation matrix
corr = numeric_df.corr()

# Heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Online Shoppers Dataset")
plt.tight_layout()
plt.show()


## Countplot for Revenue

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="Revenue", data=df)
plt.title("Distribution of Revenue")
plt.xlabel("Revenue")
plt.ylabel("Count")
plt.show()


## Countplot for Month

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(y="Month", data=df, order=df["Month"].value_counts().index)
plt.title("Number of Sessions by Month")
plt.xlabel("Count")
plt.ylabel("Month")
plt.show()


## Boxplot for PageValues by Revenue

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x="Revenue", y="PageValues", data=df)
plt.title("PageValues by Revenue")
plt.xlabel("Revenue")
plt.ylabel("PageValues")
plt.show()


 ## Scatterplot for ExitRates and PageValues

In [ ]:
sample_df = df.sample(1000, random_state=42)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=sample_df, x="ExitRates", y="PageValues", hue="Revenue")
plt.title("ExitRates vs PageValues")
plt.xlabel("ExitRates")
plt.ylabel("PageValues")
plt.show()


##  Histogram for ProductRelated_Duration

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["ProductRelated_Duration"], bins=30, kde=True)
plt.title("Distribution of ProductRelated_Duration")
plt.xlabel("ProductRelated_Duration")
plt.ylabel("Frequency")
plt.show()


## Numeric features vs Revenue

In [ ]:
df["Revenue_Label"] = df["Revenue"].map({
    0: "Revenue = False",
    1: "Revenue = True"
})


numeric_features = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]

n_cols = 3
n_rows = math.ceil(len(numeric_features) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, numeric_features):
    sns.histplot(
        data=df,
        x=col,
        hue="Revenue_Label",
        hue_order=["Revenue = False", "Revenue = True"],
        multiple="stack",
        bins=30,
        stat="density",
        common_norm=False,
        alpha=0.45,
        ax=ax
    )
    ax.set_title(f"{col} by Revenue")

for ax in axes[len(numeric_features):]:
    ax.remove()

plt.tight_layout()
plt.show()


## Categories features vs Revenue

In [ ]:
categorical_features = [
    "Month",
    "VisitorType",
    "Weekend",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType"
]

n_cols = 3
n_rows = math.ceil(len(categorical_features) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, categorical_features):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, hue="Revenue_Label", order=order, ax=ax)
    ax.set_title(f"{col} by Revenue")
    ax.tick_params(axis="x", rotation=45)

for ax in axes[len(categorical_features):]:
    ax.remove()

plt.tight_layout()
plt.show()


# Milestone 3 - The Training Loop

## Train / Validation / Test Workflow


In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_model = df.copy()
df_model["Weekend"] = df_model["Weekend"].astype(int)
df_model["Revenue"] = df_model["Revenue"].astype(int)

target_col = "Revenue"

numeric_features_model = [
    "Administrative",
    "Administrative_Duration",
    "Informational",
    "Informational_Duration",
    "ProductRelated",
    "ProductRelated_Duration",
    "BounceRates",
    "ExitRates",
    "PageValues",
    "SpecialDay"
]

categorical_features_model = [
    "Month",
    "OperatingSystems",
    "Browser",
    "Region",
    "TrafficType",
    "VisitorType",
    "Weekend"
]

feature_cols = numeric_features_model + categorical_features_model
X = df_model[feature_cols].copy()
y = df_model[target_col].copy()

# First split: keep the test set completely untouched until the final evaluation.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Second split: create the initial training set and validation set from the training portion.
# Final proportions are 60% train, 20% validation, 20% test.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    stratify=y_train_full,
    random_state=42
)

split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test", "Train + Validation"],
    "Rows": [len(X_train), len(X_val), len(X_test), len(X_train_full)],
    "Revenue = 1": [y_train.sum(), y_val.sum(), y_test.sum(), y_train_full.sum()],
    "Revenue Rate": [y_train.mean(), y_val.mean(), y_test.mean(), y_train_full.mean()]
})

display(split_summary.round(4))


### Pipelines to Prevent Preprocessing Leakage


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features_model),
        ("cat", make_ohe(), categorical_features_model)
    ]
)

preprocessor_lr = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features_model),
        ("cat", make_ohe(), categorical_features_model)
    ]
)

lr_baseline_pipe = Pipeline(steps=[
    ("preprocess", preprocessor_lr),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

rf_baseline_pipe = Pipeline(steps=[
    ("preprocess", preprocessor_tree),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1))
])

print("Pipelines are ready.")


### Initial Training Round



In [ ]:
def evaluate_model(name, model, X_train_eval, y_train_eval, X_eval, y_eval):
    train_pred = model.predict(X_train_eval)
    eval_pred = model.predict(X_eval)

    if hasattr(model, "predict_proba"):
        eval_prob = model.predict_proba(X_eval)[:, 1]
        auc = roc_auc_score(y_eval, eval_prob)
    else:
        auc = np.nan

    return {
        "Model": name,
        "Train Accuracy": accuracy_score(y_train_eval, train_pred),
        "Validation/Test Accuracy": accuracy_score(y_eval, eval_pred),
        "Validation/Test Precision": precision_score(y_eval, eval_pred, zero_division=0),
        "Validation/Test Recall": recall_score(y_eval, eval_pred, zero_division=0),
        "Validation/Test F1": f1_score(y_eval, eval_pred, zero_division=0),
        "Validation/Test ROC-AUC": auc,
        "F1 Gap": f1_score(y_train_eval, train_pred, zero_division=0) - f1_score(y_eval, eval_pred, zero_division=0)
    }

lr_baseline_pipe.fit(X_train, y_train)
rf_baseline_pipe.fit(X_train, y_train)

baseline_validation_df = pd.DataFrame([
    evaluate_model("Logistic Regression baseline", lr_baseline_pipe, X_train, y_train, X_val, y_val),
    evaluate_model("Random Forest baseline", rf_baseline_pipe, X_train, y_train, X_val, y_val)
])

display(baseline_validation_df.round(4))


# Milestone 4 - Model Optimization

### Grid Search Using the Validation Split



In [ ]:
# Combine train and validation for GridSearchCV.
# Rows marked -1 are always training rows; rows marked 0 are the validation rows.
X_grid = pd.concat([X_train, X_val], axis=0)
y_grid = pd.concat([y_train, y_val], axis=0)
validation_fold = np.concatenate([
    np.full(len(X_train), -1),
    np.zeros(len(X_val), dtype=int)
])
predefined_validation = PredefinedSplit(test_fold=validation_fold)

lr_tuning_pipe = Pipeline(steps=[
    ("preprocess", preprocessor_lr),
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

lr_param_grid = [
    {
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs"],
        "model__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        "model__class_weight": [None, "balanced"]
    },
    {
        "model__penalty": ["l1"],
        "model__solver": ["liblinear"],
        "model__C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
        "model__class_weight": [None, "balanced"]
    }
]

lr_grid = GridSearchCV(
    estimator=lr_tuning_pipe,
    param_grid=lr_param_grid,
    scoring="f1",
    cv=predefined_validation,
    refit=True,
    n_jobs=1,
    return_train_score=True
)

lr_grid.fit(X_grid, y_grid)

print("Best Logistic Regression parameters:")
print(lr_grid.best_params_)
print(f"Best validation F1: {lr_grid.best_score_:.4f}")
print("Best Logistic Regression has been refit on train + validation.")

lr_results = pd.DataFrame(lr_grid.cv_results_).sort_values("rank_test_score")
display(lr_results[[
    "rank_test_score",
    "param_model__penalty",
    "param_model__solver",
    "param_model__C",
    "param_model__class_weight",
    "mean_test_score",
    "mean_train_score"
]].head(10).round(4))


### Random Forest Overfitting Control



In [ ]:
# Random Forest overfitting control: tune max_depth first.

rf_depth_pipe = Pipeline(steps=[
    ("preprocess", preprocessor_tree),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=1
    ))
])

depth_values = [3, 5, 7, 9, 12, 15, 20, None]

rf_depth_grid = GridSearchCV(
    estimator=rf_depth_pipe,
    param_grid={"model__max_depth": depth_values},
    scoring="f1",
    cv=predefined_validation,
    refit=True,
    n_jobs=1,
    return_train_score=True
)

rf_depth_grid.fit(X_grid, y_grid)

rf_depth_results = pd.DataFrame(rf_depth_grid.cv_results_)[[
    "param_model__max_depth",
    "mean_train_score",
    "mean_test_score"
]].copy()

rf_depth_results.columns = ["max_depth", "Train F1", "Validation F1"]
rf_depth_results["F1 Gap"] = rf_depth_results["Train F1"] - rf_depth_results["Validation F1"]
rf_depth_results["Depth Label"] = rf_depth_results["max_depth"].astype(str).replace({"None": "Unlimited"})

best_depth_f1 = rf_depth_results["Validation F1"].max()
close_to_best = rf_depth_results[rf_depth_results["Validation F1"] >= best_depth_f1 - 0.01].copy()
close_to_best["Depth Sort"] = close_to_best["max_depth"].apply(lambda value: 999 if pd.isna(value) else int(value))
preferred_depth_row = close_to_best.sort_values(["Depth Sort", "F1 Gap"], ascending=[True, True]).iloc[0]
preferred_depth = preferred_depth_row["max_depth"]

print("Random Forest max_depth sweep:")
display(rf_depth_results.round(4))
print(f"Best validation F1 from depth sweep: {best_depth_f1:.4f}")
print(f"Preferred regularized depth: {preferred_depth_row['Depth Label']} "
      f"with validation F1 = {preferred_depth_row['Validation F1']:.4f} "
      f"and F1 gap = {preferred_depth_row['F1 Gap']:.4f}")

plot_depth_results = rf_depth_results.copy()
plot_depth_results["Depth Plot"] = plot_depth_results["max_depth"].apply(lambda value: 25 if pd.isna(value) else int(value))

plt.figure(figsize=(9, 5))
plt.plot(plot_depth_results["Depth Plot"], plot_depth_results["Train F1"], marker="o", label="Train F1")
plt.plot(plot_depth_results["Depth Plot"], plot_depth_results["Validation F1"], marker="s", label="Validation F1")
plt.xticks(plot_depth_results["Depth Plot"], plot_depth_results["Depth Label"])
plt.xlabel("max_depth")
plt.ylabel("F1 Score")
plt.title("Random Forest max_depth Sweep")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

if pd.isna(preferred_depth):
    refined_depths = [9, 12, 15]
else:
    preferred_depth_int = int(preferred_depth)
    all_regularized_depths = [3, 5, 7, 9, 12, 15]
    depth_index = min(range(len(all_regularized_depths)), key=lambda idx: abs(all_regularized_depths[idx] - preferred_depth_int))
    refined_depths = all_regularized_depths[max(0, depth_index - 1): min(len(all_regularized_depths), depth_index + 2)]

rf_tuning_pipe = Pipeline(steps=[
    ("preprocess", preprocessor_tree),
    ("model", RandomForestClassifier(random_state=42, n_jobs=1))
])

rf_param_grid = {
    "model__n_estimators": [300],
    "model__max_depth": refined_depths,
    "model__min_samples_leaf": [5, 10, 20],
    "model__min_samples_split": [10, 20],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": ["balanced", "balanced_subsample"]
}

rf_grid = GridSearchCV(
    estimator=rf_tuning_pipe,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=predefined_validation,
    refit=True,
    verbose=1,
    n_jobs=1,
    return_train_score=True
)

rf_grid.fit(X_grid, y_grid)

print("Best regularized Random Forest parameters:")
print(rf_grid.best_params_)
print(f"Best validation F1: {rf_grid.best_score_:.4f}")
print("Best Random Forest has been refit on train + validation.")

rf_results = pd.DataFrame(rf_grid.cv_results_).sort_values("rank_test_score")
rf_results["F1 Gap"] = rf_results["mean_train_score"] - rf_results["mean_test_score"]

display(rf_results[[
    "rank_test_score",
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__min_samples_split",
    "param_model__max_features",
    "param_model__class_weight",
    "mean_test_score",
    "mean_train_score",
    "F1 Gap"
]].head(10).round(4))


### Final Threshold Decision

The validation-tuned threshold improved precision, but it reduced recall. Because this project prioritizes identifying as many actual buyers as possible for further buying-habit analysis, the final Random Forest keeps the default `0.50` threshold.


### Final Test Evaluation



In [ ]:
def evaluate_test_model_with_threshold(name, model, threshold, X_eval, y_eval):
    eval_prob = model.predict_proba(X_eval)[:, 1]
    eval_pred = (eval_prob >= threshold).astype(int)

    return {
        "Model": name,
        "Threshold": threshold,
        "Test Accuracy": accuracy_score(y_eval, eval_pred),
        "Test Precision": precision_score(y_eval, eval_pred, zero_division=0),
        "Test Recall": recall_score(y_eval, eval_pred, zero_division=0),
        "Test F1": f1_score(y_eval, eval_pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_eval, eval_prob)
    }

# Final test evaluation.
# These are testing-split results only.
# Both models use the default 0.50 threshold.
# For Random Forest, this keeps recall higher, which matches the project goal of capturing more actual buyers.
final_model_specs = [
    ("Logistic Regression tuned", lr_grid.best_estimator_, 0.50),
    ("Random Forest tuned", rf_grid.best_estimator_, 0.50)
]

final_test_df = pd.DataFrame([
    evaluate_test_model_with_threshold(name, model, threshold, X_test, y_test)
    for name, model, threshold in final_model_specs
])

display(final_test_df.round(4))

for name, model, threshold in final_model_specs:
    test_prob = model.predict_proba(X_test)[:, 1]
    test_pred = (test_prob >= threshold).astype(int)
    cm = confusion_matrix(y_test, test_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Revenue", "Revenue"])
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"{name} - Final Test Confusion Matrix (threshold = {threshold:.2f})")
    plt.tight_layout()
    plt.show()


## Milestone 5: Final Evaluation



In [ ]:
test_target_labels = y_test.map({0: "No Revenue", 1: "Revenue"})
test_target_counts = test_target_labels.value_counts().reindex(["No Revenue", "Revenue"])
test_target_percent = (test_target_counts / test_target_counts.sum()) * 100

test_target_distribution_df = pd.DataFrame({
    "Testing Target Class": test_target_counts.index,
    "Count": test_target_counts.values,
    "Percentage": test_target_percent.values
})

display(test_target_distribution_df.round(2))

plt.figure(figsize=(7, 5))
ax = sns.barplot(
    data=test_target_distribution_df,
    x="Testing Target Class",
    y="Count",
    palette=["#4C78A8", "#F58518"]
)

for index, row in test_target_distribution_df.iterrows():
    ax.text(
        index,
        row["Count"] + 25,
        f"{int(row['Count'])}\n({row['Percentage']:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.title("Testing Split Target Variable Distribution: Revenue")
plt.xlabel("Revenue Class")
plt.ylabel("Number of Testing Sessions")
plt.ylim(0, test_target_distribution_df["Count"].max() * 1.18)
plt.tight_layout()
plt.show()


### What Each Metric Means

- **Accuracy**: Overall percentage of correct predictions. It is useful, but can be misleading because most sessions do not generate revenue.
- **Precision**: Among sessions predicted as `Revenue`, how many were actually revenue sessions. Higher precision means fewer false alarms.
- **Recall**: Among actual revenue sessions, how many were successfully detected. This is especially important here because the project aims to identify as many real buyers as possible for further buying-habit analysis.
- **F1-score**: Balance between precision and recall. It is useful when the target variable is imbalanced.
- **ROC-AUC**: Measures how well the model separates revenue and non-revenue sessions across different thresholds. Higher ROC-AUC means better ranking ability.
- **Confusion Matrix**: Shows the actual number of correct and incorrect predictions for each class, including false positives and false negatives.


In [ ]:
# Visual comparison of final test metrics.
metrics_to_plot = [
    "Test Accuracy",
    "Test Precision",
    "Test Recall",
    "Test F1",
    "Test ROC-AUC"
]

metrics_plot_df = final_test_df[["Model"] + metrics_to_plot].melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)

metrics_plot_df["Metric"] = metrics_plot_df["Metric"].str.replace("Test ", "", regex=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=metrics_plot_df,
    y="Metric",
    x="Score",
    hue="Model",
    palette=["#4C78A8", "#F58518"]
)
plt.xlim(0, 1)
plt.title("Final Model Performance on Testing Split")
plt.xlabel("Score")
plt.ylabel("Metric")
plt.legend(title="Model", loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:
rf_row = final_test_df[final_test_df["Model"].str.contains("Random Forest", case=False)].iloc[0]
lr_row = final_test_df[final_test_df["Model"].str.contains("Logistic Regression", case=False)].iloc[0]

print("Final Testing Split Analysis")
print("- The test split was used only once, after model tuning was completed.")
print(f"- Random Forest test recall: {rf_row['Test Recall']:.4f}")
print(f"- Logistic Regression test recall: {lr_row['Test Recall']:.4f}")
print(f"- Random Forest test F1-score: {rf_row['Test F1']:.4f}")
print(f"- Logistic Regression test F1-score: {lr_row['Test F1']:.4f}")
print(f"- Random Forest ROC-AUC: {rf_row['Test ROC-AUC']:.4f}")
print(f"- Logistic Regression ROC-AUC: {lr_row['Test ROC-AUC']:.4f}")


### Final Model Decision

For this project, **Random Forest** is the preferred final model. The reason is not only that it performs well numerically, but that its behaviour matches the business and analysis objective. Since the next step is to analyse buying habits, it is more important to capture a larger portion of actual buyers than to be overly conservative.

